In [2]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, RMSprop, SGD



In [32]:
BASE_DIR = Path.cwd().parent
DATA_DIR     = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "dl_features"
MODELS_DIR   = FEATURES_DIR

print("BASE_DIR      :", BASE_DIR)
print("FEATURES_DIR  :", FEATURES_DIR)

BASE_DIR      : C:\Users\DELL\nlp-fake-news-detector-transformers
FEATURES_DIR  : C:\Users\DELL\nlp-fake-news-detector-transformers\data\dl_features


In [4]:
def load_features():
    print("Loading preprocessed features...")

    X_train = np.load(FEATURES_DIR / "X_train_pad.npy")
    X_val   = np.load(FEATURES_DIR / "X_val_pad.npy")
    X_test  = np.load(FEATURES_DIR / "X_test_pad.npy")

    y_train = np.load(FEATURES_DIR / "y_train.npy")
    y_val   = np.load(FEATURES_DIR / "y_val.npy")
    y_test  = np.load(FEATURES_DIR / "y_test.npy")

    with open(FEATURES_DIR / "meta.pkl", "rb") as f:
        meta = pickle.load(f)

    with open(FEATURES_DIR / "tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)

    print(f"  X_train shape : {X_train.shape}")
    print(f"  X_val   shape : {X_val.shape}")
    print(f"  X_test  shape : {X_test.shape}")
    print(f"  max_len       : {meta['max_len']}")
    print(f"  max_words     : {meta['max_words']}")
    print(f"  Vocab size    : {len(tokenizer.word_index)}")

    return X_train, X_val, X_test, y_train, y_val, y_test, meta, tokenizer

In [10]:
import gensim.downloader as api

def load_glove_gensim():
    print("Loading GloVe (gensim)...")
    return api.load("glove-wiki-gigaword-100")


In [12]:
def build_embedding_matrix_gensim(tokenizer, model, max_words, embedding_dim):
    embedding_matrix = np.zeros((max_words, embedding_dim))

    for word, i in tokenizer.word_index.items():
        if i >= max_words:
            continue

        if word in model:
            embedding_matrix[i] = model[word]

    return embedding_matrix

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

def build_model_with_pretrained(max_words, max_len, embedding_dim, embedding_matrix, optimizer):

    model = Sequential([
        Input(shape=(max_len,)),

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable=False
        ),

        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),

        Dense(64, activation='relu'),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy']
    )

    return model

In [14]:
def train_and_evaluate(model, name, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\nTraining {name}...")

    early_stop = EarlyStopping(patience=3, restore_best_weights=True)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=128,
        callbacks=[early_stop],
        verbose=1
    )

    preds = (model.predict(X_test) > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    print(f"\n{name} Results:")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)

    return acc, f1

In [15]:
optimizers = {
    "Adam": Adam(learning_rate=0.001),
    "RMSprop": RMSprop(learning_rate=0.001),
    "SGD": SGD(learning_rate=0.01, momentum=0.9),
}

In [16]:
glove_model = load_glove_gensim()

Loading GloVe (gensim)...


In [17]:
X_train, X_val, X_test, y_train, y_val, y_test, meta, tokenizer = load_features()

max_words = meta['max_words']
max_len   = meta['max_len']

results = {}
embedding_dim_glove = 100

embedding_matrix_glove = build_embedding_matrix_gensim(
    tokenizer, glove_model, max_words, embedding_dim_glove
)

for opt_name, opt in optimizers.items():

    print(f"\n=== GloVe + {opt_name} ===\n")

    model = build_model_with_pretrained(
        max_words,
        max_len,
        embedding_dim_glove,
        embedding_matrix_glove,
        optimizer=opt
    )

    results[f"GloVe_{opt_name}"] = train_and_evaluate(
        model,
        f"GloVe + {opt_name}",
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )

Loading preprocessed features...
  X_train shape : (963423, 20)
  X_val   shape : (222329, 20)
  X_test  shape : (296438, 20)
  max_len       : 20
  max_words     : 20000
  Vocab size    : 295251

=== GloVe + Adam ===


Training GloVe + Adam...
Epoch 1/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 153s 20ms/step - accuracy: 0.7464 - loss: 0.5161 - val_accuracy: 0.7635 - val_loss: 0.4884
Epoch 2/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 212s 21ms/step - accuracy: 0.7690 - loss: 0.4839 - val_accuracy: 0.7686 - val_loss: 0.4795
Epoch 3/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 252s 28ms/step - accuracy: 0.7770 - loss: 0.4707 - val_accuracy: 0.7714 - val_loss: 0.4776
Epoch 4/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 160s 21ms/step - accuracy: 0.7823 - loss: 0.4618 - val_accuracy: 0.7738 - val_loss: 0.4776
Epoch 5/5
7527/7527 ━━━━━━━━━━━━━━━━━━━━ 158s 21ms/step - accuracy: 0.7858 - loss: 0.4552 - val_accuracy: 0.7738 - val_loss: 0.4736
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 69s 7ms/step

GloVe + Adam Results:
Accuracy : 0.772623617754808

In [20]:
results

{'GloVe_Adam': (0.7726236177548088, 0.7714710978053386),
 'GloVe_RMSprop': (0.7724448282608842, 0.7738333925661676),
 'GloVe_SGD': (0.7722694121536375, 0.7631846660071703)}

In [31]:
import pandas as pd
comparison = []

for name, res in results.items():
    acc, f1 = res

    comparison.append({
        "model": name,
        "accuracy": acc,
        "f1": f1
    })

comparison.append({
    "model": "Baseline_CNN",
    "accuracy": 0.7935,
    "f1": 0.7949
})

df = pd.DataFrame(comparison)
df = df.sort_values("accuracy", ascending=False)

print(df)

           model  accuracy        f1
3   Baseline_CNN  0.793500  0.794900
0     GloVe_Adam  0.772624  0.771471
1  GloVe_RMSprop  0.772445  0.773833
2      GloVe_SGD  0.772269  0.763185
